In [ ]:
%pip install sentencepiece

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
import shutil
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

os.makedirs("models", exist_ok=True)
os.makedirs("models/checkpoints", exist_ok=True)
os.makedirs("data", exist_ok=True)

input_path = "/kaggle/input/my-tokenizer/my_tokenizer.model"
output_path = "models/my_tokenizer.model"

if os.path.exists(input_path):
    print(f"Copying tokenizer from {input_path}...")
    shutil.copy(input_path, output_path)   
else:
    print(f"Tokenizer not found at {input_path}, a new one will be trained.")

In [ ]:
from dataclasses import dataclass
import math

# AGGRESSIVE TRAINING CONFIG

@dataclass
class Config:
    n_embd = 768
    n_head = 12
    n_layer = 16
    mini_batch_size = 8 # B
    context_length = 2048 # T
    total_batch_size = 8*mini_batch_size*context_length # MUST BE DIVISBLE by (B*T)
    device = 'cuda'
    vocab_size = 32768
max_iter = 14000
config = Config()

def get_lr(step):
    max_lr = 1e-3
    min_lr = 1e-4
    warmup_steps=500
    max_steps=14000
    if step < warmup_steps:
        return max_lr * (step+1)/warmup_steps 
    if step > max_steps:
        return min_lr
    decay_ratio = (step-warmup_steps)/(max_steps-warmup_steps)
    coefficient = 0.5 * (1.0 + math.cos(decay_ratio * math.pi)) # starts at 1 reduces to 0
    return min_lr + coefficient * (max_lr - min_lr)

In [ ]:
import torch
import sentencepiece as spm

# --------------------module.py---------------------------
from typing import Any
from abc import abstractmethod, ABC

class Module(ABC):
    
    @abstractmethod
    def forward(self, x, **kwargs):
        pass    
    
    def __init__(self) -> None:
        super().__init__()
        self.buffers = dict()
        self.is_train = True
    
    def parameters(self):
        params = []
        def try_append_attribute(attr):    
            if isinstance(attr, torch.Tensor) and attr.requires_grad:
                params.append(attr)
                    
            if isinstance(attr, Module):
                params.extend(attr.parameters())
                
            if isinstance(attr, list):
                for item in attr:
                    try_append_attribute(item)
        for attribute in self.__dict__.values():
            try_append_attribute(attribute)
                    
        return params
    
    def state_dict(self, destination = None, prefix = ''):
        
        if destination is None:
            destination = dict()
        
        def handle_item(attr, current_key):
            if isinstance(attr, torch.Tensor) and attr.requires_grad:
                destination[current_key] = attr.detach()
                
            if isinstance(attr, Module):
                attr.state_dict(destination, current_key + '.')  
            
            if isinstance(attr, list):
                for i, item in enumerate(attr):      
                    handle_item(item, f'{current_key}.{i}')
            
        for name, value in self.__dict__.items():
            handle_item(value, prefix+name)
    
        return destination
    
    def load_state_dict(self, state_dict: dict, prefix = ''):
        
        def handle_item(attr, current_key):
            if isinstance(attr, torch.Tensor) and attr.requires_grad:
                if current_key in state_dict:
                    loaded_value = state_dict[current_key]
                    if attr.shape != loaded_value.shape:
                        print(f"Shape mismatch for {current_key}: expected {attr.shape}, got {loaded_value.shape}")
                    attr.data.copy_(loaded_value)
                else:
                    print(f"Key {current_key} not found in state_dict")
                
            if isinstance(attr, Module):
                attr.load_state_dict(state_dict, current_key + '.')  
            
            if isinstance(attr, list):
                for i, item in enumerate(attr):      
                    handle_item(item, f'{current_key}.{i}')
            
        for name, value in self.__dict__.items():
            handle_item(value, prefix+name)
    
    def to(self, device, dtype):
        def list_to(list):
            for i, item in enumerate(list):
                if isinstance(item, torch.Tensor):
                    was_param = item.requires_grad
                    moved = item.to(device=device, dtype=dtype)
                    if was_param:
                        moved = Param(moved)
                    list[i] = moved
                if isinstance(item, Module):
                    list[i] = item.to(device, dtype)
                if isinstance(item, list):
                    list_to(item)
        for key in self.__dict__.keys():
            attribute = getattr(self, key)
        
            if isinstance(attribute, torch.Tensor):
                was_param = attribute.requires_grad
                moved = attribute.to(device=device, dtype=dtype)
                if was_param:
                    moved = Param(moved)
                setattr(self, key, moved)
            if isinstance(attribute, Module):
                setattr(self, key,attribute.to(device, dtype))
            if isinstance(attribute, list):
                list_to(attribute)
        
        return self
        
    def __call__(self, x, **kwargs)->Any:
        return self.forward(x, **kwargs)
    
    def train(self):
        self._set_train_(True)
        return self
    def eval(self):
        self._set_train_(False)
        return self
        
    def _set_train_(self, new_train):
        self.is_train = new_train
        def recursive_set_train(attr):                
            if isinstance(attr, Module):
                attr._set_train_(new_train)
            if isinstance(attr, list):
                for item in attr:
                    recursive_set_train(item)
        for key in self.__dict__.keys():
            attribute = getattr(self, key)
            recursive_set_train(attribute)
    
def Param(tensor: torch.Tensor) -> torch.Tensor:
    return tensor.requires_grad_(True)

# ----------------------rope.py-----------------------------
def precompute_rotary_embeddings(head_size: int, context_length: int, theta: float = 10000.0, device = 'cpu'):

    assert head_size%2 == 0, "RoPE only supports even number of dimensions of q, k."
    powers = torch.arange(0, head_size, 2, dtype=torch.float32, device=device) / head_size # (0 to 1), steps of 2 cus we're pairing
    
    freqs = 1.0 / (theta ** powers)
    
    t = torch.arange(context_length,dtype=torch.float32,  device=device) 
    
    # outer product makes a matrix of size (context_length, hs//2) to get the angle for each position based on its corresponding frequency
    freqs = torch.outer(t, freqs)
    
    cos, sin = freqs.cos(), freqs.sin()
    cos, sin = cos[None,None,:,:], sin[None,None,:, :] # to be broadcasted later as (B_b, nh_b, T, hs//2)
    
    return cos, sin

def apply_rope(x, cos, sin):
    assert x.ndim == 4 # must be (B, nh, T, hs)
    x_fp32 = x.float()
    half = x_fp32.shape[-1]//2
    x1, x2 = x_fp32[..., :half], x_fp32[..., half:] # (B, nh, T, hs//2)
    # for a pair (x1, x2), where it'll be like (x1, x33) incase of dim=64, after rotating the new coords (X, Y):
    # its that table i learned in JEE, 
    # X = x1*cost - x2*sint | [cos, -sin]
    # Y = x1*sint + x2*cost | [sin,  cos]
    y1 = x1 * cos - x2 * sin # Y
    y2 = x1 * (sin) + x2 * cos # X
    # # cat along the last dimension (hs//2)
    return torch.cat([y1, y2], dim=3).type_as(x) # (B, nh, T, hs)



# ----------------------layers.py--------------------------------
class Linear(Module):
    
    def __init__(self, n_in, n_out, dtype = torch.float32, device = 'cpu', bias = False):
        super().__init__()
        kwargs = {'dtype': dtype, 'device': device}
        self.weight: torch.Tensor = Param(torch.randn(n_in, n_out, **kwargs) * (n_in ** -0.5)) # scaling down with * 1/sqrt(fan_in). gain isn't really required due to normalization layers.
        self.bias = Param(torch.zeros(n_out, **kwargs)) if bias else None
        
    def forward(self, x):
        x = x @ self.weight # (B, T, C_in) @ (C_in, C_out) => (B, T, C_out)
        if self.bias is not None:
            x = x + self.bias # (B, T, C_out) + (T_broadcasted, C_out)
        return x

class Embedding(Module):
    
    def __init__(self, n_in, n_out, dtype = torch.float32, device = 'cpu'):
        super().__init__()
        kwargs = {'dtype': dtype, 'device': device}
        self.embedding = Param(torch.randn(n_in, n_out, **kwargs) * n_out**-0.5) 
        
    def forward(self, x):
        return self.embedding[x] # (vocab_size, n_embd) indexed with (B, T) gives (B, T, n_embd) where values of x at [B, T] are within [0, vocab_size)
    
class SiLU(Module):
    def forward(self, x):
        return x * torch.sigmoid(x) # modern replacement for reLU (precursor to swiGLU, might add that later. idk for small models though)
    
    
class LayerNorm(Module):
    
    def __init__(self, n_in, eps = 1e-8,dtype = torch.float32, device = 'cpu'):
        super().__init__()
        kwargs = {'dtype': dtype, 'device': device}
        self.eps = eps
        self.gamma = Param(torch.ones(n_in, **kwargs)) # (n_embd, )
        self.beta = Param(torch.zeros(n_in, **kwargs)) # (n_embd, ) but will be broadcasted to (n_embd_b, n_embd) during addition
    
    def forward(self, x):
        
        mean = x.mean(-1, keepdim = True) # (B, T, 1)
        var = x.var(-1, keepdim = True, unbiased = False) # (B, T, 1)
        x = (x - mean) * (var + self.eps)**-0.5 # ((B, T, C) - (B, T, C_b)) * (B, T, C_b) 
        x = self.gamma * x + self.beta # (B_b, T_b, C) * (B, T, C) + (B_b, T_b, C)
        return x
       
class RMSNorm(Module):
    
    def __init__(self, n_in, eps = 1e-8,dtype = torch.float32, device = 'cpu'):
        super().__init__()
        kwargs = {'dtype': dtype, 'device': device}
        self.eps = eps
        self.gamma = Param(torch.ones(n_in, **kwargs)) # (n_embd, )
        # no beta!
    
    def forward(self, x: torch.Tensor):
        
        # RMS: square, then mean, then root. eg. sqrt((x2 + y2 + z2)/3)
        x_fp32 = x.float() # we cast to fp32 so when I square and add during mean etc we have higher accuracy in such large values
        square = x_fp32.pow(2) # (B, T, C) all values of all tokens in all batches squared
        mean = square.mean(dim=-1, keepdim=True) # (B, T, 1) all embeddings for each token of each batch are mean'd. 
        # we keepdim so that when i divide later we don't have the issue of (B, T, C) / (B, T) [would get C vs T not equal issue]
        root_reciprocal = torch.rsqrt(mean + self.eps) # r_sqrt is just faster for reciprocals
        
        return self.gamma * (x_fp32 * root_reciprocal).type_as(x) # gamma*(x/rms(x)), type is casted back to normal (bf16 probably)      
        
class MultiHeadAttention(Module):
    def __init__(self, config: Config, dtype = torch.float32, device = 'cpu'):
        super().__init__()
        self.kwargs = {'dtype': dtype, 'device': device}
        assert config.n_embd % config.n_head == 0, "n_embd needs to be an integer multiple of n_heads."
        self.n_embd = config.n_embd
        self.n_head = config.n_head
        self.qkv = Linear(config.n_embd, config.n_embd * 3, **self.kwargs) # q, k, v in a single matrix (C, 3*C)
        self.tril = torch.tril(torch.ones(config.context_length, config.context_length, **self.kwargs)).view(1,1,config.context_length, config.context_length)
        # so that later i can use this as (B_b, nh_b, T, T)
        # as this mask is applied to attention scores, (B, nh, T, T) 
        self.out_proj = Linear(config.n_embd, config.n_embd, **self.kwargs)
        
    """cos_sin is just (cos, sin) tuple"""
    def forward(self, x, cos_sin):
        
        B,T,C = x.shape # C == n_embd
        qkv = self.qkv(x) # gives (B, T, C) @ (C, C*3) => (B, T, C*3)
        q,k,v = torch.split(qkv, self.n_embd, dim=-1) # split back along the (C*3) to get (B,T,C)
        head_size = self.n_embd // self.n_head # head_size * n_head = n_embd
        q = q.view(B, T, self.n_head, head_size).transpose(1,2) # view (B,T,C) as (B,T,nh,hs) => (B,nh,T,hs) where nh*hs = C 
        k = k.view(B, T, self.n_head, head_size).transpose(1,2) # (B,nh,T,hs) 
        v = v.view(B, T, self.n_head, head_size).transpose(1,2) # (B,nh,T,hs) 
        
        cos, sin = cos_sin
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        
        # attention = (q @ k.transpose(-2, -1)) * head_size**-0.5 # (B, nh, T, hs) @ (B, nh, hs, T) => (B, nh, T, T) 
        # attention = attention.masked_fill(self.tril[:, :, :T, :T] == 0, float('-inf')) # the == 0 creates a boolean mask of (1, 1, T, T)
        # attention = torch.softmax(attention, dim = -1) # we want each row in (T,T) to sum to 1 as it acts like probabilities of how useful other tokens are for each token
        # result = attention @ v # (B, nh, T, T) @ (B, nh, T, hs) => (B, nh, T, hs) each token focuses on highly aligned other tokens to weigh in their embeddings
        result = torch.nn.functional.scaled_dot_product_attention(q, k, v, is_causal=True)
        result = result.transpose(1,2).contiguous().view(B, T, C) # (B, nh, T, hs) => (B, T, nh, hs), make them contiguous so we can view them as (B, T, C)
        result = self.out_proj(result) # (B, T, C) @ (C, C) => (B, T, C)
        return result
        
class Sequential(Module):
    def __init__(self, layers: list) -> None:
        super().__init__()
        self.layers = layers
    
    def forward(self, x, **kwargs):
        for layer in self.layers:
            x = layer(x, **kwargs)
        return x
        
class FeedForward(Module):
    def __init__(self, config: Config, dtype = torch.float32, device = 'cpu'):
        super().__init__()
        self.kwargs = {'dtype': dtype, 'device': device}
        self.mlp = Sequential([
            Linear(config.n_embd, config.n_embd * 4, **self.kwargs),
            SiLU(),
            Linear(config.n_embd*4, config.n_embd, **self.kwargs)
        ])
    
    def forward(self, x):
        x = self.mlp(x)
        return x
    
class SwiGLU(Module):
        def __init__(self, config: Config, dtype = torch.float32, device = 'cpu'):
            super().__init__()
            self.kwargs = {'dtype': dtype, 'device': device}
            # to be equivalent to C*4 from ffwd, we require 8/3*C as now there's 3 layers rather than 2, keeping params same.
            n_hidden = nearest_multiple(int(8 * config.n_embd / 3), divisor=256) # closest multiple of 256 to 8/3*C to make it a 'nice-number' 
            # (multiple works fine, Pow of 2 would require too big of a difference in size for not much in effeciency.)
            self.gate_value_proj = Linear(config.n_embd, n_hidden * 2, **self.kwargs)
            self.silu = SiLU()
            self.out_proj = Linear(n_hidden, config.n_embd, **self.kwargs)
        
        def forward(self, x):
            # Swiglu = Silu(wx) * (w'x) (element-wise multiplication)
            combined = self.gate_value_proj(x) # (B, T, n_hidden*2)
            x1, x2 = combined.chunk(chunks=2, dim=-1) # split (B, T, n_hidden * 2) along dim = -1 into 2 pieces
            gate = self.silu(x1) # (B, T, n_hidden)
            value = x2 # (B, T, n_hidden)
            return self.out_proj(gate * value) # (B, T, n_hidden) @ (n_hidden, C) => (B, T, C)

def nearest_multiple(x, divisor):
    return ((x+divisor-1)//divisor) * divisor

# ----------------------------model.py-----------------------------------w
class Block(Module):
    def __init__(self, config: Config, dtype = torch.float32, device = 'cpu'):
        super().__init__()
        self.attention = MultiHeadAttention(config, dtype, device)
        self.feed_forward = SwiGLU(config, dtype, device) # i've updated from linear MLP to SwiGLU, but its still considered an ffwd layer.
        self.rmsn_1 = RMSNorm(config.n_embd, 1e-8, dtype, device)
        self.rmsn_2 = RMSNorm(config.n_embd, 1e-8, dtype, device)
        
    def forward(self, x, cos_sin):
        x = x + self.attention(self.rmsn_1(x), cos_sin=cos_sin)
        x = x + self.feed_forward(self.rmsn_2(x))
        return x

class Transformer(Module):
    def __init__(self, config: Config, dtype = torch.float32, device = 'cpu'):
        super().__init__()
        self.config = config
        self.token_embedding = Embedding(config.vocab_size, config.n_embd, dtype, device)
        #self.pos_embedding = Embedding(config.context_length, config.n_embd, dtype, device)
        self.blocks = Sequential([
            Block(config, dtype, device) for _ in range(config.n_layer)
        ])
        self.rmsn_final = RMSNorm(config.n_embd, 1e-8, dtype, device)
        self.cos, self.sin = precompute_rotary_embeddings(config.n_embd // config.n_head, config.context_length, device=config.device)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        token_embedding = self.token_embedding(idx) # (B, T) => (B, T, C)
        # REPLACING THIS WITH ROPE
        # pos_embedding = self.pos_embedding(torch.arange(0, T, device = idx.device)) # (T,C)
        x = token_embedding # + pos_embedding # (B, T, C) + (B_b, T, C)
        x = self.blocks(x, cos_sin = (self.cos[:, :, :T, :], self.sin[:, :, :T, :]))
        x = self.rmsn_final(x)
        logits = x @ self.token_embedding.embedding.T # lm_head, but directly applied (B, T, C) @ (C, vocab_size) => (B, T, vocab_size)
        if targets is not None:
            logits_flat = logits.view(B*T, -1)
            targets = targets.view(B*T)
            loss = torch.nn.functional.cross_entropy(logits_flat, targets)
        else: 
            loss = torch.zeros((1))
        return logits, loss
    

    """
    exclusively adamw for all optimization. configure_adamw and configure_muon when using them combined.
    """
    def configure_optimizer(self, weight_decay, learning_rate, device):
        params = self.parameters()
        decay_params = [p for p in params if p.ndim >= 2]
        non_decay_params = [p for p in params if p.ndim < 2] # don't decay the biases or 1D vectors
        optim_groups = [
            {'params': decay_params, 'weight_decay': weight_decay},
            {'params': non_decay_params, 'weight_decay': 0.0},
        ]
        return torch.optim.AdamW(optim_groups, lr=learning_rate, eps=1e-8, fused='cuda' in device)
    
    def configure_adamw(self, weight_decay, learning_rate, device):
        params = self.parameters()
        non_decay_params = [p for p in params if p.ndim < 2] # don't decay the biases or 1D vectors
        decay_params = self.token_embedding.parameters()
        optim_groups=[
            {'params': non_decay_params, 'weight_decay': 0.0},
            {'params': decay_params, 'weight_decay': weight_decay}
        ]
        return torch.optim.AdamW(optim_groups, lr=learning_rate, eps=1e-8, fused='cuda' in device)
    
    def configure_muon(self, weight_decay, learning_rate, device):
        params = self.parameters()
        embedding_params = self.token_embedding.parameters()# my .parameters() is not a generator, but a list.
        embd_ids = {id(p) for p in embedding_params}
        hidden_weights = [p for p in params if p.ndim >= 2 and id(p) not in embd_ids]
        
        return torch.optim.Muon(hidden_weights, lr=learning_rate, weight_decay=weight_decay, eps=1e-7, adjust_lr_fn="match_rms_adamw")
 
@torch.no_grad()
def generate(model, idx, max_new_tokens):
    for i in range(max_new_tokens):
        idx_cut = idx[:, -model.config.context_length:]
        logits, _ = model(idx_cut) # (B, T, C)
        logits = logits[:, -1, :] # last token, (B, C)
        probs = torch.softmax(logits, dim = -1)
        idx_new = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, idx_new], dim=-1) 
    return idx 
 
@torch.no_grad()   
def prompt(prompt: str, model, tokenizer: spm.SentencePieceProcessor, new_tokens = 256, debug=False, device = 'cpu'):
    model.eval()
    tokens = generate(model, torch.tensor([[tokenizer.eos_id()] + tokenizer.Encode(prompt)], dtype=torch.long, device=device), new_tokens) # (B, T), but B = 1
    tokens = tokens[0].tolist() # (T,), converted from tensor to pylist
    model.train()
    return ''.join(tokenizer.IdToPiece(id) for id in tokens).replace('▁', ' ').replace('</s>', '</s>\n') if debug else tokenizer.Decode(tokens)

    

# ----------------------------remotedataset.py----------------------------------
from typing import Iterator
import torch
import sentencepiece as spm
from datasets import load_dataset, interleave_datasets
from torch.utils.data import IterableDataset

SHUFFLE_BUFFER_SIZE = 1000

class PackedDataset(IterableDataset):
    def __init__(self, hf_dataset, tokenizer: spm.SentencePieceProcessor, B, T) -> None:
        self.hf_dataset = hf_dataset
        self.context_length = T
        self.B = B
        self.sp = tokenizer
        self.eos_id = tokenizer.PieceToId('<EOS>')
        
    def tokenize_and_group(self, iterable, column="text"):
        iterator = iter(iterable)
        
        buffer = []
        block_size = self.B * self.context_length + 1
        for text_example in iterator:
            tokens = self.sp.EncodeAsIds(text_example[column])
            if len(tokens) == 0: continue
            tokens.append(self.eos_id)
            buffer.extend(tokens)
            
            while len(buffer) >= block_size:
                data= torch.tensor(buffer[:block_size], dtype=torch.long)
                xs = data[:-1].view(self.B, self.context_length) # (B, T)
                ys = data[1:].view(self.B, self.context_length) # (B, T)
                yield xs, ys
                buffer=buffer[block_size-1:]
                
    def __iter__(self) -> Iterator:
        worker_info = torch.utils.data.get_worker_info()
        if worker_info is None:
            return self.tokenize_and_group(self.hf_dataset)
        else:
            sharded_dataset = self.hf_dataset.shard(num_shards = worker_info.num_workers, index=worker_info.id)
            return self.tokenize_and_group(sharded_dataset)
    
def get_interleaved_dataset():
    smol_data_sources = [
        "cosmopedia-v2",
        "fineweb-edu-dedup",
    ]
    
    hf_datasources = []
    for source in smol_data_sources:
        ds = load_dataset("HuggingFaceTB/smollm-corpus", source, split='train', streaming=True)
        hf_datasources.append(ds)
        
    ds = load_dataset("transformersbook/codeparrot", split='train',streaming=True)
    ds=ds.rename_column('content', 'text')
    hf_datasources.append(ds)        
    
    probs = [0.75, 0.15, 0.1] # cosmopedia, fineweb-edu, python 
    mixed_dataset = interleave_datasets(hf_datasources, probs, seed=42, stopping_strategy="all_exhausted")
    
    return mixed_dataset.shuffle(buffer_size=SHUFFLE_BUFFER_SIZE, seed=42)

def train_tokenizer(input_path, vocab_size = 32768, max_samples = 250_000):
            
    sp = spm.SentencePieceTrainer.Train(
        input=input_path,
        model_prefix = 'models/my_tokenizer',
        vocab_size=vocab_size,
        model_type='bpe',
        character_coverage = 0.9995,
        input_sentence_size = max_samples,
        shuffle_input_sentence = True,
        byte_fallback = True,
        split_digits = True,
        allow_whitespace_only_pieces = True,
        remove_extra_whitespaces = False,
        user_defined_symbols = ['<start_of_turn>', '<end_of_turn>', '<EOS>'],
        pad_id = 0,
        unk_id = 1,
        bos_id = 2,
        eos_id = 3, # won't really be using this but keeping it in for safety
    )
    
def create_tokenizer_corpus(dataset, max_samples = 250_000):
    with open('data/token-corpus.txt', 'w', encoding='utf-8') as f:
        for idx, text_example in enumerate(dataset):
            text = text_example["text"]

            if text is not None:
                f.write(text)
            else: print(f"Text was none at {idx}; most likely one or more of the interleaved datasets don't have 'text' column")
            if idx >= max_samples:
                break
    return 'data/token-corpus.txt'

In [ ]:
import torch.nn.functional as F
import time
import sentencepiece as spm
torch.manual_seed(42)

In [ ]:
TOKENIZER_DIR = "models/my_tokenizer.model"
CHECKPOINT_DIR = "models/checkpoints/"


B = config.mini_batch_size
T = config.context_length
grad_accum_steps = config.total_batch_size // (B*T)
assert config.total_batch_size % (B * T) == 0, "Total batch size must be divisble by (B*T)"

In [ ]:
import sentencepiece as spm

if not os.path.exists(TOKENIZER_DIR):
    print(f"Tokenizer not found at {TOKENIZER_DIR}. Training new...")
    hf_dataset = get_interleaved_dataset()
    train_tokenizer(create_tokenizer_corpus(hf_dataset)) 
    
sp = spm.SentencePieceProcessor(model_file = TOKENIZER_DIR) # pyright: ignore[reportCallIssue]

In [ ]:
hf_dataset = get_interleaved_dataset()
val_hf = hf_dataset.take(1000)
train_hf = hf_dataset.skip(1000)

train_ds = PackedDataset(train_hf, sp, config.mini_batch_size, config.context_length)
val_ds = PackedDataset(val_hf, sp, config.mini_batch_size, config.context_length)
train_dataloader = torch.utils.data.DataLoader(train_ds, batch_size=None, num_workers=2, pin_memory=True,prefetch_factor=4, persistent_workers=True)
val_dataloader = torch.utils.data.DataLoader(val_ds, batch_size=None, num_workers=0, pin_memory=True)# prefetch_factor=4, persistent_workers=True)
def infinite_dataloader(dataloader):
    while True:
        for batch in dataloader:
            return batch
train_dataloader = iter(infinite_dataloader(train_dataloader))

In [ ]:
print(sp.Encode("Lily found a needle.")) # pyright: ignore[reportAttributeAccessIssue]
print(sp.Decode(sp.Encode("Lily found a needle.")))

In [ ]:
torch.set_float32_matmul_precision('high')
model = Transformer(config, dtype=torch.float32, device=config.device)
params = model.parameters()

In [ ]:
print(sum(p.numel() for p in params)/1e6, "M params")

In [ ]:
optimizer_adamw = model.configure_optimizer(weight_decay=0.1, learning_rate=get_lr(0), device=config.device)

In [ ]:
checkpoint_path = os.path.join(CHECKPOINT_DIR, "checkpoint_latest.pt")
if os.path.exists(checkpoint_path):
    print(f"Found checkpoint at {checkpoint_path}, loading...")
    checkpoint = torch.load(checkpoint_path, map_location=config.device, weights_only=False)
    
    model.load_state_dict(checkpoint['model'])
    optimizer_adamw.load_state_dict(checkpoint['optimizer'])
    step = checkpoint['step']
    last_val_loss = checkpoint['val_loss']
    print(f"Checkpoint loaded. Continuing from step {step}")
else:
    step = 0
    last_val_loss = 0.0

In [ ]:
model_optim = torch.compile(model)

In [ ]:
def get_val_loss():
    max_batches = 10
    val_loss = []
    with torch.no_grad():
        for i, (xs, ys) in enumerate(val_dataloader):      
            if i >= max_batches:
                break
            xs = xs.to(device=config.device, non_blocking=True)
            ys= ys.to(device=config.device, non_blocking=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                _, loss = model_optim(xs, targets=ys)
            val_loss.append(loss.item())
    return sum(val_loss)/len(val_loss)

In [ ]:
data_iter = iter(train_dataloader)
lossi = []
timei = []
mean_stat_batches = 50
print_batches = 50
val_batches = 500
checkpoint_batches = 250
permanent_checkpoint_batches = 2000

scalar = torch.amp.GradScaler('cuda') # pyright: ignore[reportPrivateImportUsage]
try:
    for _ in range(step, max_iter):
        t_start = time.time()
        optimizer_adamw.zero_grad(set_to_none=True)
        
        # forward pass
        loss_accum = torch.tensor(0.0, device=config.device)
        for micro_step in range(grad_accum_steps):
            xs, ys= next(data_iter) # loop_around=True so we don't need to worry about StopIteration exception
            xs = xs.to(device=config.device, non_blocking=True)
            ys= ys.to(device=config.device, non_blocking=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                logits, loss = model_optim(xs, targets=ys)
            loss = loss / grad_accum_steps # normalizer to avoid loss gradients being exagerrated by grad_accum_steps times.
            loss_accum += loss.detach()
            # backward pass
            scalar.scale(loss).backward()
        
        scalar.unscale_(optimizer_adamw)
        norm = torch.nn.utils.clip_grad_norm_(params, 1.0)
        
        # update
        lr = get_lr(step)
        for param_group in optimizer_adamw.param_groups:
            param_group['lr'] = lr
            
        scalar.step(optimizer_adamw)
        
        scalar.update()
        
        torch.cuda.synchronize()
        t_end = time.time()
        
        # track stats    
        lossi.append(loss_accum.item())
        timei.append(t_end-t_start)
        if step % print_batches == 0:
            lossi = lossi[-mean_stat_batches:]
            timei = timei[-mean_stat_batches:]
            train_loss = sum(lossi)/len(lossi)
            if step % val_batches == 0:
                val_loss = get_val_loss()
                last_val_loss = val_loss
            else: 
                val_loss = 0.0
            dt = sum(timei)/len(timei)
            tokens_per_sec = (xs.shape[0] * xs.shape[1] * grad_accum_steps)/dt # (B*T)/dt
            print(f"Step {step}: Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | dt: {dt*1000:.2f}ms | {int(tokens_per_sec):,d} tok/s")
        
        step += 1
        
        # checkpointing, after current step is OVER. hence next batch should continue fron NEXT STEP.
        if step > 0 and (step-1) % checkpoint_batches == 0:
            checkpoint = {
                'model': model.state_dict(),
                'optimizer': optimizer_adamw.state_dict(),
                'config': config,
                'step': step,
                'val_loss': last_val_loss
            }
            
            torch.save(checkpoint, os.path.join(CHECKPOINT_DIR, "checkpoint_latest.pt"))
            print(f"Checkpoint saved for step {step}.")
            if (step-1) % permanent_checkpoint_batches == 0:
                torch.save(checkpoint, os.path.join(CHECKPOINT_DIR, f"checkpoint_step_{step}.pt"))
                
finally:
    checkpoint = {
                'model': model.state_dict(),
                'optimizer': optimizer_adamw.state_dict(),
                'config': config,
                'step': step,
                'val_loss': last_val_loss
            }
            
    torch.save(checkpoint, os.path.join(CHECKPOINT_DIR, "checkpoint_finally.pt"))
    print(f"Checkpoint saved for step {step} as 'checkpoint_finally.pt'")

In [ ]:
val_loss = get_val_loss()
val_loss

In [ ]:
print(prompt("Tim loves to eat toast. So he asked his mom to cook some", model=model, tokenizer=sp, new_tokens=256, debug=True, device=config.device))